# Social Media Sentiment Analysis Using Big Data
### Comprehensive Big Data Computation (BDC) Exploration Notebook

This notebook walks through the complete Big Data pipeline:
1. **Dataset Ingestion & Inspection:** Sentiment140 (1.6M rows), US Airline Tweets (3 classes), SQLite database.
2. **Data Cleaning & Preprocessing:** Regex normalization, contraction expansion, stopword filtering with negation retention.
3. **Big Data MapReduce Simulation:** Multi-worker parallel Map, Combiner, Shuffle & Sort, and Reduce.
4. **Feature Extraction:** TF-IDF Vectorization with unigrams and bigrams.
5. **Model Benchmarking:** Multinomial Naive Bayes Baseline vs. Logistic Regression.
6. **Evaluation & Inference:** Accuracy, Precision, Recall, F1-Score, and Confusion Matrix.

In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to sys.path
project_root = Path('.').resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data_processing.load_dataset import load_sentiment140, load_airline_tweets, load_sqlite_tweets, get_dataset_summary
from src.data_processing.preprocess import clean_text, tokenize
from src.big_data.mapreduce_simulator import MapReduceEngine, word_count_mapper, sum_reducer
from src.models.predict import predict_text, predict_dataframe

print('Environment and modules loaded successfully!')

## 1. Dataset Ingestion and Schema Inspection

In [ ]:
# Load Sentiment140 Sample (5,000 balanced records)
df_s140 = load_sentiment140(sample_size=5000)
print('Sentiment140 Shape:', df_s140.shape)
print(df_s140['sentiment_label'].value_counts())
df_s140.head(3)

In [ ]:
# Load US Airline Tweets
df_air = load_airline_tweets(sample_size=3000)
print('Airline Tweets Shape:', df_air.shape)
print(df_air['sentiment'].value_counts())
df_air.head(3)

## 2. Text Preprocessing & Cleaning

In [ ]:
sample_raw = '@switchfoot http://twitpic.com/2y1zl - Awww, that is a bummer! I can not wait for #summer :D'
sample_cleaned = clean_text(sample_raw, remove_stopwords=True)
print('RAW TEXT:    ', sample_raw)
print('CLEANED TEXT:', sample_cleaned)

## 3. Parallel MapReduce Demonstration

In [ ]:
# Run multi-worker MapReduce word count
sample_corpus = df_s140['text'].head(1000).tolist()
mr_engine = MapReduceEngine(num_mappers=4, num_reducers=2, use_combiner=True)
mr_results = mr_engine.run_job(sample_corpus, word_count_mapper, sum_reducer)

print(f'Total Records Processed: {mr_results["total_input_records"]}')
print(f'Mappers: {mr_results["num_mappers"]} | Reducers: {mr_results["num_reducers"]}')
print(f'Execution Time: {mr_results["total_time_seconds"]*1000:.2f} ms')

# Top 10 words from MapReduce
sorted_words = sorted(mr_results['results'].items(), key=lambda x: x[1], reverse=True)[:10]
pd.DataFrame(sorted_words, columns=['Word', 'Count'])

## 4. Model Inference & Sentiment Predictions

In [ ]:
test_statements = [
    'This big data computation project is working exceptionally well!',
    'Horrible experience, flight delayed 6 hours and lost my baggage.',
    'I am genuinely impressed by the speed and reliability of Apache Spark and MapReduce.'
]

for text in test_statements:
    res = predict_text(text)
    print(f'Text: {res["original_text"]}')
    print(f'Sentiment: {res["sentiment"].upper()} (Confidence: {res["confidence"]*100:.2f}%)')
    print(f'Probabilities: {res["probabilities"]}\n')